# 06A1 Amber/OpenMM: polymer in water

GAFF2 polymer (ABCG2 charges, from `05A_amber_gaff2_parameters.ipynb`) · OPC water + Na⁺/Cl⁻ (ion parameters from `leaprc.water.opc`) · OpenMM.

This notebook builds one PHA oligomer in a water box with AmberTools `tleap` and prepares staged OpenMM MD. The polymer-only benchmark uses this system type: the box (1.2 nm padding, cubic, 0.15 M salt after neutralising) and the MD protocol are the benchmark's, as in `06B1_charmm_gromacs_polymer_in_water.ipynb`; only the non-bonded settings are Amber's. For the enzyme + polymer system, use `06A2_amber_openmm_enzyme_polymer_in_water.ipynb`.

| Stage (file prefix) | Settings |
| --- | --- |
| `step6.0_minimization` | tolerance 1000 kJ mol⁻¹ nm⁻¹, up to 50000 iterations |
| `step6.1_nvt` | 100 ps, 2 fs, 300 K (Langevin, 1 ps⁻¹) |
| `step6.2_npt` | 500 ps, 2 fs, 300 K, 1 bar (Monte Carlo barostat) |
| `step7_production` | 100 ns, 2 fs, NPT, frames every 2 ps |
| all stages | PME, 1.0 nm cut-off (Amber), HBonds constraints, no restraints |

Every step that runs `tleap` or OpenMM is behind a `RUN_…` flag that defaults to `False`.

## Caveat: GAFF2 with OPC

GAFF2 (and its ABCG2/AM1-BCC charges) was not parameterised against the OPC water model. GAFF2 + OPC (+ ff19SB for proteins) is a common combination, and ff19SB is recommended with OPC, but it has **not been validated for PHA**. Treat hydration-dependent results (solvation free energies, polymer–water interactions) with that in mind. The enzyme–PHA production runs used the CHARMM/GROMACS route (CHARMM36m + CGenFF + CHARMM TIP3P; notebooks 06B1/06D), not this route.

## §0 Optional quick parameter check (no solvent)

A sanity check of the 05A GAFF2 files before building the water box: the topology loads, the polymer minimises and survives a few hundred steps of MD in OpenMM. It is not a physical result (no solvent, no periodic environment). Its run flag `RUN_OPENMM_DRY` is `False` by default; skip this section if you only want the water box.

### Scope

- Engine: OpenMM
- System: dry polymer, usually no periodic solvent box
- Input: `gaff2/<SYSTEM>.prmtop` and `gaff2/<SYSTEM>.inpcrd`
- Output: `examples/output/md_tests/<SYSTEM>/openmm/dry_polymer/`
- Stages: minimisation, short NVT, optional NPT only if the input already has box vectors, short production

The simplified dry polymer workflow remains the default validation route for small PHA oligomers.


In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd() if (Path.cwd() / "src" / "iphasimulator").exists() else Path.cwd().parent
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

output_root = repo_root / "examples" / "output"
from iphasimulator.naming import oligomer_name

system_name = oligomer_name("3HB", 4)
md_root = output_root / "md_tests" / system_name
print(f"Repository: {repo_root}")
print(f"System: {system_name}")
print(f"MD output root: {md_root}")

gaff2_dir = md_root / "gaff2"
openmm_dry_dir = md_root / "openmm" / "dry_polymer"
prmtop_path = gaff2_dir / f"{system_name}.prmtop"
inpcrd_path = gaff2_dir / f"{system_name}.inpcrd"

{
    "prmtop": prmtop_path,
    "prmtop_exists": prmtop_path.exists(),
    "inpcrd": inpcrd_path,
    "inpcrd_exists": inpcrd_path.exists(),
    "openmm_dry_dir": openmm_dry_dir,
}

Repository: /Users/k20098771/opt/iPHASimulator_v2
System: P3HB_4
MD output root: /Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/P3HB_4


{'prmtop': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/P3HB_4/gaff2/P3HB_4.prmtop'),
 'prmtop_exists': True,
 'inpcrd': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/P3HB_4/gaff2/P3HB_4.inpcrd'),
 'inpcrd_exists': True,
 'openmm_dry_dir': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/P3HB_4/openmm/dry_polymer')}

### Check OpenMM

This only checks that the current Python environment can import OpenMM.


In [2]:
from iphasimulator.simulation_openmm_amber_runner import openmm_available

openmm_available()

True

### Short local settings

These settings are intentionally small. Increase them only after the quick check runs cleanly.

In [3]:
openmm_settings = {
    "minimization_max_iterations": 200,
    "nvt_steps": 100,
    "npt_steps": 100,
    "production_steps": 100,
    "report_interval": 10,
    "temperature_kelvin": 300.0,
    "pressure_bar": 1.0,
    "platform_name": None,  # set to "CUDA" for a local GPU check
    "platform_precision": "mixed",
}

openmm_settings

{'minimization_max_iterations': 200,
 'nvt_steps': 100,
 'npt_steps': 100,
 'production_steps': 100,
 'report_interval': 10,
 'temperature_kelvin': 300.0,
 'pressure_bar': 1.0,
 'platform_name': None,
 'platform_precision': 'mixed'}

### Run the quick check

This cell is disabled by default so running the notebook never creates trajectories or logs. Set `RUN_OPENMM_DRY = True` when the GAFF2 inputs exist and you are ready to write local output files.

In [6]:
from iphasimulator.simulation_openmm_amber_runner import run_openmm_with_amber_topology

RUN_OPENMM_DRY = False

if RUN_OPENMM_DRY:
    openmm_dry_outputs = run_openmm_with_amber_topology(
        prmtop_path,
        inpcrd_path,
        openmm_dry_dir,
        **openmm_settings,
    )
    openmm_dry_outputs
else:
    print("Set RUN_OPENMM_DRY = True to run the dry OpenMM validation workflow.")

### Expected outputs

Generated trajectories, checkpoints, and logs live under `examples/output/`, which is ignored by Git by default.


In [7]:
expected_outputs = [
    "minimized.pdb",
    "nvt.dcd",
    "nvt.log",
    "npt.log",
    "production.dcd",
    "production.log",
    "state.xml",
    "final.pdb",
    "openmm_summary.log",
]

{filename: (openmm_dry_dir / filename).exists() for filename in expected_outputs}

{'minimized.pdb': True,
 'nvt.dcd': True,
 'nvt.log': True,
 'npt.log': True,
 'production.dcd': True,
 'production.log': True,
 'state.xml': True,
 'final.pdb': True,
 'openmm_summary.log': True}

## 1. Inputs

- `SYSTEM_NAME`, `MOL2_PATH`, `FRCMOD_PATH`: the GAFF2 files from 05A (`<SYSTEM>.gaff2.mol2`, `<SYSTEM>.gaff2.frcmod`).
- `OUTPUT_DIR`: a **new** folder for the system.
- Box and salt: `padding_nm` 1.2 (cubic), `salt_molar` 0.15, as in the benchmark.
- `JOB_NAME`: the SLURM job name.

In [ ]:
from pathlib import Path
import sys

repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "iphasimulator").exists())
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))
from iphasimulator.amber_solvation import (
    POLYMER_IN_WATER_PROTOCOL,
    AmberSolvationSettings,
    build_solvated_amber_system,
    run_openmm_short_test,
    run_openmm_stages,
    write_openmm_run_files,
    write_tleap_input,
)

SYSTEM_NAME = "P3HB_4"
gaff2_dir = repo_root / "examples/output/md_tests" / SYSTEM_NAME / "gaff2"
MOL2_PATH = gaff2_dir / f"{SYSTEM_NAME}.gaff2.mol2"
FRCMOD_PATH = gaff2_dir / f"{SYSTEM_NAME}.gaff2.frcmod"
OUTPUT_DIR = repo_root / "examples/output/md_tests" / SYSTEM_NAME / "openmm" / "polymer_in_water"
JOB_NAME = f"{SYSTEM_NAME}_amber_polymer_MD"

settings = AmberSolvationSettings(padding_nm=1.2, box_shape="box", cubic=True, salt_molar=0.15)
protocol = POLYMER_IN_WATER_PROTOCOL
print("mol2 found:  ", MOL2_PATH.is_file(), MOL2_PATH)
print("frcmod found:", FRCMOD_PATH.is_file(), FRCMOD_PATH)
print("output:      ", OUTPUT_DIR, "(exists — choose a new folder)" if OUTPUT_DIR.exists() else "")

## 2. Build polymer in water (tleap, OPC, ions)

`tleap` runs twice in `OUTPUT_DIR`: first to count the waters for the padding, then to neutralise, add `round(salt × waters / 55.5)` Na⁺/Cl⁻ pairs and save `system.prmtop`, `system.inpcrd` and `system.pdb`, with `tleap.in` and `tleap.log`. The ion parameter file that `leaprc.water.opc` loaded is printed. The preview shows the tleap input without running anything.

Writing the run files puts `run_openmm_md.py` (stand-alone; needs only OpenMM), `protocol.json`, `run_step6_local.sh` and `run_hpc_equilibration_production.slurm` next to `system.prmtop`, with the same stage names as the GROMACS folders: step 6.0 locally, then 6.1, 6.2 and 7 on the cluster.

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as preview_dir:
    print(write_tleap_input(Path(preview_dir) / "tleap.in", mol2_name=MOL2_PATH.name,
                            frcmod_name=FRCMOD_PATH.name, settings=settings, salt_pairs=0))

RUN_TLEAP = False

if not RUN_TLEAP:
    print("Set RUN_TLEAP = True to build the polymer in water with tleap.")
else:
    system = build_solvated_amber_system(MOL2_PATH, FRCMOD_PATH, OUTPUT_DIR, settings=settings)
    print("Waters added:        ", system.water_count)
    print("Salt ion pairs added:", system.salt_ion_pairs)
    print("Ion parameter files: ", ", ".join(system.ion_parameter_files) or "none reported")
    for path in write_openmm_run_files(OUTPUT_DIR, protocol, job_name=JOB_NAME, polymer_residues=("PHA",)):
        print("Wrote", path)

## 3. OpenMM short test

Minimisation, then 10 ps NVT and 10 ps NPT with the protocol's temperature, pressure and time steps, written to `OUTPUT_DIR/short_test/`. It reports the potential energy and density after each stage; water boxes should approach about 1.0 g/mL. Ten picoseconds only show that the system is stable, not that it is equilibrated.

In [ ]:
RUN_OPENMM_TEST = False

if not RUN_OPENMM_TEST:
    print("Set RUN_OPENMM_TEST = True to run the short OpenMM test.")
else:
    for result in run_openmm_short_test(OUTPUT_DIR, platform_name=None):  # e.g. "CUDA"
        print(f"{result['stage']}: Epot {result['potential_energy_kj_mol']:.1f} kJ/mol, density {result['density_g_ml']:.3f} g/mL")

## 4. Production files for 07

The run files were written in step 2. Run step 6.0 (minimisation) locally if you like, then hand the folder to `07_hpc_execution.ipynb` (OpenMM section), which submits `run_hpc_equilibration_production.slurm` for steps 6.1, 6.2 and 7.

In [ ]:
RUN_LOCAL_MINIMIZATION = False

if not RUN_LOCAL_MINIMIZATION:
    print("Set RUN_LOCAL_MINIMIZATION = True to run step 6.0 locally.")
else:
    print(run_openmm_stages(OUTPUT_DIR, ["step6.0_minimization"]))

for name in ("system.prmtop", "run_openmm_md.py", "protocol.json", "step6.0_minimization.xml", "run_hpc_equilibration_production.slurm"):
    print(f"{'found  ' if (OUTPUT_DIR / name).exists() else 'missing'}  {name}")
print(f"On the cluster: cd {OUTPUT_DIR} && sbatch run_hpc_equilibration_production.slurm")